<a href="https://colab.research.google.com/github/sudulran/IT3051-Group27-Hotel-Cancellation-Prediction/blob/main/notebooks/02_Preprocessing_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hotel Booking Cancellation Prediction

## Notebook 02 — Data Preprocessing and Feature Engineering

**Module:** IT3051 – Fundamentals of Data Mining  
**Group:** Group 27 – Entropy Zero

### Objective

This notebook prepares the Hotel Booking Demand dataset for machine-learning model development.

The preprocessing decisions are based on the findings obtained during exploratory data analysis in Notebook 01.

The main tasks include:

- defining the prediction point;
- preventing data leakage;
- handling timing-sensitive variables;
- handling missing values;
- performing feature engineering;
- selecting suitable predictive features;
- defining numerical and categorical preprocessing;
- creating a leakage-safe train/test split;
- and preparing a reusable preprocessing pipeline.

### Prediction Task

**Target:** `is_canceled`

- `0` — Not Cancelled
- `1` — Cancelled

**Machine-learning task:** Binary classification

####Import libraries

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

####Connect to GitHub repository

In [2]:
REPO_NAME = "IT3051-Group27-Hotel-Cancellation-Prediction"
REPO_URL = f"https://github.com/sudulran/{REPO_NAME}.git"
REPO_PATH = f"/content/{REPO_NAME}"

if not os.path.exists(REPO_PATH):
    !git clone {REPO_URL}
else:
    print("Repository already cloned.")

%cd {REPO_PATH}

Cloning into 'IT3051-Group27-Hotel-Cancellation-Prediction'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 94 (delta 38), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 1.70 MiB | 2.85 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/IT3051-Group27-Hotel-Cancellation-Prediction


####Load raw data

In [3]:
DATA_PATH = "data/raw/hotel_bookings.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

Dataset loaded successfully.
Dataset shape: (119390, 32)


####Protect the raw dataframe

In [4]:
data = df.copy()

print("Working copy created.")
print("Raw dataset shape:", df.shape)
print("Working dataset shape:", data.shape)

Working copy created.
Raw dataset shape: (119390, 32)
Working dataset shape: (119390, 32)


## 1. Define the Prediction Point

The intended system predicts whether a reservation is likely to be cancelled at or shortly after the booking is created, before the final reservation outcome and later booking events are known.

Therefore, predictive variables should represent information that would reasonably be available at this prediction point.

Variables that directly reveal the final outcome or depend on events occurring later in the reservation lifecycle must be excluded to prevent data leakage.


## 2. Construct Temporal Variables

The dataset does not provide a direct booking-creation date. However, `lead_time` represents the number of days between the date the reservation entered the hotel system and the arrival date.

Therefore, an approximate booking date can be reconstructed as:

**Booking Date = Arrival Date − Lead Time**

This reconstructed booking date is used only to create a chronological train/test split. It will not be used directly as a predictive feature.

####Construct arrival date and booking date

In [5]:
data["arrival_date"] = pd.to_datetime(
    data["arrival_date_year"].astype(str)
    + "-"
    + data["arrival_date_month"]
    + "-"
    + data["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d"
)

data["booking_date"] = (
    data["arrival_date"]
    - pd.to_timedelta(data["lead_time"], unit="D")
)

print("Arrival date range:")
print(data["arrival_date"].min(), "to", data["arrival_date"].max())

print("\nReconstructed booking date range:")
print(data["booking_date"].min(), "to", data["booking_date"].max())

Arrival date range:
2015-07-01 00:00:00 to 2017-08-31 00:00:00

Reconstructed booking date range:
2013-06-24 00:00:00 to 2017-08-31 00:00:00


## 3. Chronological Train/Test Split

A chronological holdout is used as the primary train/test strategy because the intended system will make predictions for future bookings.

Approximately the earliest 80% of bookings, ordered by reconstructed booking date, are assigned to the training set. The remaining later bookings form the test set.

The split is performed using complete dates rather than cutting through records from the same date.

The test set will be treated as a final holdout and will not be used to make ambiguous preprocessing decisions.

####Determine an approximately 80% chronological cutoff

In [6]:
date_counts = (
    data.groupby("booking_date")
    .size()
    .sort_index()
)

cumulative_proportion = date_counts.cumsum() / len(data)

eligible_dates = cumulative_proportion[
    cumulative_proportion <= 0.80
]

cutoff_date = eligible_dates.index[-1]

print("Chronological cutoff date:", cutoff_date)
print(
    "Cumulative proportion at cutoff:",
    round(cumulative_proportion.loc[cutoff_date] * 100, 2),
    "%"
)

Chronological cutoff date: 2017-01-11 00:00:00
Cumulative proportion at cutoff: 79.91 %


#### Create chronological training and test sets

In [7]:
train_data = data[
    data["booking_date"] <= cutoff_date
].copy()

test_data = data[
    data["booking_date"] > cutoff_date
].copy()

print("Training records:", len(train_data))
print("Test records:", len(test_data))

print(
    f"\nTraining proportion: {len(train_data) / len(data) * 100:.2f}%"
)
print(
    f"Test proportion: {len(test_data) / len(data) * 100:.2f}%"
)

print("\nLatest training booking date:")
print(train_data["booking_date"].max())

print("\nEarliest test booking date:")
print(test_data["booking_date"].min())

Training records: 95401
Test records: 23989

Training proportion: 79.91%
Test proportion: 20.09%

Latest training booking date:
2017-01-11 00:00:00

Earliest test booking date:
2017-01-12 00:00:00


#### Verify temporal separation

In [8]:
assert (
    train_data["booking_date"].max()
    < test_data["booking_date"].min()
)

print("Chronological separation verified successfully.")

Chronological separation verified successfully.


#### Target distribution after splitting

In [9]:
split_target_summary = pd.DataFrame({
    "Training (%)": (
        train_data["is_canceled"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ),
    "Test (%)": (
        test_data["is_canceled"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
    )
}).round(2)

split_target_summary.index = [
    "Not Cancelled (0)",
    "Cancelled (1)"
]

split_target_summary

,Training (%),Test (%)
Not Cancelled (0),61.57,68.48
Cancelled (1),38.43,31.52


### Test-Set Lock

From this point onward, ambiguous preprocessing decisions are investigated using the training set only.

The test set is retained as a chronological holdout and will not be used to decide whether unusual observations should be removed, retained, transformed, or otherwise treated.

Preprocessing operations learned from data, such as imputation and scaling, will be fitted using training data only and then applied unchanged to the test set.

## 4. Training-Only Investigation of Ambiguous Data-Quality Cases

EDA identified several situations where automatic deletion or modification would not be justified.

These cases are examined using only the training data before preprocessing decisions are finalized.

The investigation considers:

- missing values;
- exact duplicate records;
- zero-guest bookings;
- zero-night bookings;
- negative and very high ADR values;
- and large-group bookings.

### Missing values in training data

In [10]:
training_missing = pd.DataFrame({
    "Missing Count": train_data.isnull().sum(),
    "Missing Percentage": (
        train_data.isnull().mean() * 100
    ).round(3)
})

training_missing = training_missing[
    training_missing["Missing Count"] > 0
].sort_values(
    "Missing Percentage",
    ascending=False
)

training_missing

,Missing Count,Missing Percentage
company,90308,94.661
agent,12409,13.007
country,415,0.435
children,4,0.004


### Investigate missing country

In [11]:
country_missing_analysis = (
    train_data
    .assign(
        country_missing=train_data["country"].isna()
    )
    .groupby("country_missing")["is_canceled"]
    .agg(["count", "mean"])
)

country_missing_analysis[
    "Cancellation Rate (%)"
] = (
    country_missing_analysis["mean"] * 100
).round(2)

country_missing_analysis

,count,mean,Cancellation Rate (%)
country_missing,,,
False,94986,0.385373,38.54
True,415,0.139759,13.98


### Check training-set mode for children

In [12]:
children_mode = train_data["children"].mode().iloc[0]

print("Most frequent children value:", children_mode)
print(
    "Number of missing children values in training:",
    train_data["children"].isna().sum()
)

Most frequent children value: 0.0
Number of missing children values in training: 4


### Missing-Value Treatment Decision

The following treatments are selected:

- Missing `agent` means no travel agent is associated with the booking. A binary `has_agent` feature will therefore be created and the raw identifier removed.
- Missing `company` means no company is associated with the booking. A binary `has_company` feature will therefore be created and the raw identifier removed.
- Missing `country` values will be represented by an `Unknown` category rather than assigning an arbitrary country.
- Missing `children` values will be imputed using the most frequent value learned from the training data.

The actual imputation operations will be included inside the preprocessing pipeline so that information from the test set cannot influence the learned preprocessing parameters.

### Investigate duplicate records in training

In [13]:
duplicate_check_data = train_data.drop(
    columns=["arrival_date", "booking_date"]
)

duplicate_extra_count = (
    duplicate_check_data
    .duplicated()
    .sum()
)

duplicate_involved_mask = (
    duplicate_check_data
    .duplicated(keep=False)
)

duplicate_involved_count = duplicate_involved_mask.sum()

print(
    "Additional exact duplicate rows:",
    duplicate_extra_count
)

print(
    "All rows involved in duplicate groups:",
    duplicate_involved_count
)

print(
    f"Percentage involved in duplicate groups: "
    f"{duplicate_involved_count / len(train_data) * 100:.2f}%"
)

Additional exact duplicate rows: 28935
All rows involved in duplicate groups: 35786
Percentage involved in duplicate groups: 37.51%


### Compare duplicate-group cancellation rates

In [14]:
duplicate_analysis = pd.DataFrame({
    "Group": [
        "Rows involved in duplicate groups",
        "Rows not involved in duplicate groups"
    ],
    "Count": [
        duplicate_involved_mask.sum(),
        (~duplicate_involved_mask).sum()
    ],
    "Cancellation Rate (%)": [
        train_data.loc[
            duplicate_involved_mask,
            "is_canceled"
        ].mean() * 100,

        train_data.loc[
            ~duplicate_involved_mask,
            "is_canceled"
        ].mean() * 100
    ]
})

duplicate_analysis[
    "Cancellation Rate (%)"
] = duplicate_analysis[
    "Cancellation Rate (%)"
].round(2)

duplicate_analysis

,Group,Count,Cancellation Rate (%)
0,Rows involved in duplicate groups,35786,59.10
1,Rows not involved in duplicate groups,59615,26.03


### Duplicate Treatment Decision

Exact duplicate rows will be retained.

The dataset does not contain a unique booking identifier, so identical records cannot be proven to represent accidental duplicate entries. Multiple genuine reservations can share the same recorded characteristics.

In addition, removing all duplicate rows would discard a substantial proportion of the dataset.

Therefore, the existence of identical observations alone is not considered sufficient evidence for deletion.

The chronological split also reduces the risk that identical reservations from the same booking period are distributed randomly between the training and final test sets.

### Investigate unusual observations using training data

In [15]:
zero_guest_mask = (
    (train_data["adults"] == 0)
    & (train_data["children"] == 0)
    & (train_data["babies"] == 0)
)

zero_night_mask = (
    (train_data["stays_in_weekend_nights"] == 0)
    & (train_data["stays_in_week_nights"] == 0)
)

negative_adr_mask = (
    train_data["adr"] < 0
)

very_high_adr_mask = (
    train_data["adr"] > 1000
)

large_group_mask = (
    train_data["adults"] > 10
)

conditions = {
    "Zero recorded guests": zero_guest_mask,
    "Zero recorded nights": zero_night_mask,
    "Negative ADR": negative_adr_mask,
    "ADR > 1000": very_high_adr_mask,
    "More than 10 adults": large_group_mask
}

rows = []

for condition_name, mask in conditions.items():

    count = int(mask.sum())

    cancellation_rate = (
        train_data.loc[
            mask,
            "is_canceled"
        ].mean() * 100
        if count > 0
        else np.nan
    )

    other_rate = (
        train_data.loc[
            ~mask,
            "is_canceled"
        ].mean() * 100
    )

    rows.append({
        "Observation": condition_name,
        "Count": count,
        "Percentage of Training Data": round(
            count / len(train_data) * 100,
            3
        ),
        "Cancellation Rate (%)": round(
            cancellation_rate,
            2
        ) if count > 0 else np.nan,
        "Other Records Cancellation Rate (%)":
            round(other_rate, 2)
    })

unusual_training_summary = pd.DataFrame(rows)

unusual_training_summary

,Observation,Count,Percentage of Training Data,Cancellation Rate (%),Other Records Cancellation Rate (%)
0,Zero recorded guests,123,0.129,13.01,38.46
1,Zero recorded nights,599,0.628,4.17,38.65
2,Negative ADR,1,0.001,0.00,38.43
3,ADR > 1000,1,0.001,100.00,38.43
4,More than 10 adults,12,0.013,100.00,38.42


### Inspect large-group adult counts

In [16]:
train_data.loc[
    large_group_mask,
    [
        "hotel",
        "adults",
        "children",
        "babies",
        "lead_time",
        "market_segment",
        "customer_type",
        "adr",
        "is_canceled"
    ]
].sort_values(
    "adults",
    ascending=False
)

,hotel,adults,children,babies,lead_time,market_segment,customer_type,adr,is_canceled
2173,Resort Hotel,55,0.0,0,338,Direct,Group,0.0,1
1643,Resort Hotel,50,0.0,0,336,Direct,Group,0.0,1
1539,Resort Hotel,40,0.0,0,304,Direct,Group,0.0,1
1917,Resort Hotel,27,0.0,0,349,Direct,Group,0.0,1
1962,Resort Hotel,27,0.0,0,352,Direct,Group,0.0,1
1587,Resort Hotel,26,0.0,0,333,Offline TA/TO,Group,0.0,1
1752,Resort Hotel,26,0.0,0,340,Offline TA/TO,Group,0.0,1
1884,Resort Hotel,26,0.0,0,347,Offline TA/TO,Group,0.0,1
2003,Resort Hotel,26,0.0,0,354,Offline TA/TO,Group,0.0,1
2164,Resort Hotel,26,0.0,0,361,Offline TA/TO,Group,0.0,1


### Inspect ADR extremes

In [17]:
train_data.loc[
    negative_adr_mask | very_high_adr_mask,
    [
        "hotel",
        "lead_time",
        "adults",
        "children",
        "babies",
        "stays_in_weekend_nights",
        "stays_in_week_nights",
        "adr",
        "customer_type",
        "market_segment",
        "is_canceled"
    ]
]

,hotel,lead_time,adults,children,babies,stays_in_weekend_nights,stays_in_week_nights,adr,customer_type,market_segment,is_canceled
14969,Resort Hotel,195,2,0.0,0,4,6,-6.38,Transient-Party,Groups,0
48515,City Hotel,35,2,0.0,0,0,1,5400.00,Transient,Offline TA/TO,1


### Unusual-Observation Treatment Decision

The unusual observations identified during EDA will be retained.

The investigation does not provide sufficient evidence that these records are erroneous:

- Zero-guest records may represent special operational or group-related bookings.
- Zero-night bookings form a legitimate-looking operational pattern rather than isolated random corruption.
- Large adult counts may represent group or block reservations.
- The negative and extremely high ADR observations are rare, but rarity alone is not sufficient evidence to alter or delete them.
- Exact values should not be manually corrected without evidence of what the correct values should have been.

Instead of arbitrarily deleting or capping these observations, robust scaling will be used for numerical variables. The effect of extreme observations can also be reconsidered later during model evaluation if necessary.

### Treatment decision summary

In [18]:
treatment_decisions = pd.DataFrame({
    "Issue": [
        "Missing agent",
        "Missing company",
        "Missing country",
        "Missing children",
        "Exact duplicates",
        "Zero-guest bookings",
        "Zero-night bookings",
        "Negative ADR",
        "Very high ADR",
        "Large-group bookings"
    ],

    "Treatment": [
        "Create has_agent and remove raw agent ID",
        "Create has_company and remove raw company ID",
        "Encode missing values as Unknown",
        "Most-frequent imputation using training data",
        "Keep",
        "Keep",
        "Keep",
        "Keep",
        "Keep",
        "Keep"
    ],

    "Reason": [
        "Missing indicates no agent; raw ID is identifier-like",
        "Missing indicates no company; raw ID is identifier-like",
        "Avoid inventing a specific country",
        "Only a very small number are missing",
        "No unique booking ID proves they are erroneous",
        "Can represent legitimate special booking patterns",
        "Can represent legitimate operational/day-use patterns",
        "Unusual but not proven erroneous",
        "Extreme but not proven erroneous",
        "Consistent with possible group reservations"
    ]
})

treatment_decisions

,Issue,Treatment,Reason
0,Missing agent,Create has_agent and remove raw agent ID,Missing indicates no agent; raw ID is identifi...
1,Missing company,Create has_company and remove raw company ID,Missing indicates no company; raw ID is identi...
2,Missing country,Encode missing values as Unknown,Avoid inventing a specific country
3,Missing children,Most-frequent imputation using training data,Only a very small number are missing
4,Exact duplicates,Keep,No unique booking ID proves they are erroneous
5,Zero-guest bookings,Keep,Can represent legitimate special booking patterns
6,Zero-night bookings,Keep,Can represent legitimate operational/day-use p...
7,Negative ADR,Keep,Unusual but not proven erroneous
8,Very high ADR,Keep,Extreme but not proven erroneous
9,Large-group bookings,Keep,Consistent with possible group reservations


## 5. Remove Leakage and Timing-Sensitive Variables

The prediction point is at or shortly after booking creation.

The following variables are excluded from the predictive feature set:

- `reservation_status` — directly reveals the final outcome;
- `reservation_status_date` — represents when the final reservation status was established;
- `assigned_room_type` — may depend on hotel decisions made after the original reservation;
- `booking_changes` — contains information about changes occurring after the original booking;
- `days_in_waiting_list` — may depend on events occurring after the reservation was created.

These columns are removed from both the training and test feature sets using the same deterministic transformation.

## 6. Feature Engineering

Several additional features are created from information available at the prediction point:

- `total_nights` — total weekend and weekday nights;
- `total_guests` — total number of adults, children, and babies;
- `has_children` — whether children or babies are included;
- `has_agent` — whether the reservation involves a travel agent;
- `has_company` — whether the reservation is associated with a company;
- `is_zero_guest` — whether the booking records zero guests;
- `is_zero_night` — whether the booking records zero nights;
- `booking_month` — month in which the reservation was created.

The original detailed guest and stay variables are retained because they may contain information beyond their totals. Feature importance and model performance can later determine whether further feature reduction is beneficial.

#### Obtain training-derived value used for engineered guest features

In [19]:
children_fill_value = (
    train_data["children"]
    .mode()
    .iloc[0]
)

print(
    "Training-derived children fill value:",
    children_fill_value
)

Training-derived children fill value: 0.0


#### Feature-engineering function

In [20]:
def engineer_features(frame, children_fill_value):

    result = frame.copy()

    # Use training-derived value only for engineered features.
    children_for_features = (
        result["children"]
        .fillna(children_fill_value)
    )

    result["total_nights"] = (
        result["stays_in_weekend_nights"]
        + result["stays_in_week_nights"]
    )

    result["total_guests"] = (
        result["adults"]
        + children_for_features
        + result["babies"]
    )

    result["has_children"] = (
        (
            children_for_features
            + result["babies"]
        ) > 0
    ).astype(int)

    result["has_agent"] = (
        result["agent"].notna()
    ).astype(int)

    result["has_company"] = (
        result["company"].notna()
    ).astype(int)

    result["is_zero_guest"] = (
        result["total_guests"] == 0
    ).astype(int)

    result["is_zero_night"] = (
        result["total_nights"] == 0
    ).astype(int)

    result["booking_month"] = (
        result["booking_date"]
        .dt.month
        .astype(str)
    )

    return result

#### Apply feature engineering separately

In [21]:
train_prepared = engineer_features(
    train_data,
    children_fill_value
)

test_prepared = engineer_features(
    test_data,
    children_fill_value
)

print(
    "Training shape after feature engineering:",
    train_prepared.shape
)

print(
    "Test shape after feature engineering:",
    test_prepared.shape
)

Training shape after feature engineering: (95401, 42)
Test shape after feature engineering: (23989, 42)


#### Remove leakage, identifier and helper columns

In [22]:
columns_to_remove = [
    # Direct leakage / later information
    "reservation_status",
    "reservation_status_date",
    "assigned_room_type",
    "booking_changes",
    "days_in_waiting_list",

    # Raw identifier-like fields
    "agent",
    "company",

    # Helper dates used for splitting/engineering
    "arrival_date",
    "booking_date"
]

train_prepared = train_prepared.drop(
    columns=columns_to_remove,
    errors="ignore"
)

test_prepared = test_prepared.drop(
    columns=columns_to_remove,
    errors="ignore"
)

print("Removed columns:")

for column in columns_to_remove:
    print("-", column)

print(
    "\nTraining shape:",
    train_prepared.shape
)

print(
    "Test shape:",
    test_prepared.shape
)

Removed columns:
- reservation_status
- reservation_status_date
- assigned_room_type
- booking_changes
- days_in_waiting_list
- agent
- company
- arrival_date
- booking_date

Training shape: (95401, 33)
Test shape: (23989, 33)


#### Verify removed columns

In [23]:
remaining_removed_columns = [
    column
    for column in columns_to_remove
    if column in train_prepared.columns
]

if len(remaining_removed_columns) == 0:
    print(
        "All leakage, identifier, and helper "
        "columns were removed successfully."
    )
else:
    print(
        "WARNING — columns still present:",
        remaining_removed_columns
    )

All leakage, identifier, and helper columns were removed successfully.


#### Verify that no rows were deleted

In [24]:
print(
    "Training rows before preparation:",
    len(train_data)
)

print(
    "Training rows after preparation:",
    len(train_prepared)
)

print(
    "\nTest rows before preparation:",
    len(test_data)
)

print(
    "Test rows after preparation:",
    len(test_prepared)
)

assert len(train_data) == len(train_prepared)
assert len(test_data) == len(test_prepared)

print(
    "\nNo rows were removed during preprocessing."
)

Training rows before preparation: 95401
Training rows after preparation: 95401

Test rows before preparation: 23989
Test rows after preparation: 23989

No rows were removed during preprocessing.


## 7. Separate Predictors and Target

The target variable `is_canceled` is separated from the predictor variables.

The chronological test set remains untouched by any learned preprocessing operation.

#### Separate X and y

In [25]:
X_train = train_prepared.drop(
    columns=["is_canceled"]
)

y_train = train_prepared[
    "is_canceled"
].copy()

X_test = test_prepared.drop(
    columns=["is_canceled"]
)

y_test = test_prepared[
    "is_canceled"
].copy()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nX_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (95401, 32)
y_train shape: (95401,)

X_test shape: (23989, 32)
y_test shape: (23989,)


#### Verify predictor consistency

In [26]:
assert list(X_train.columns) == list(X_test.columns)

print(
    "Training and test predictor columns "
    "are identical and in the same order."
)

Training and test predictor columns are identical and in the same order.


#### Missing values remaining before pipeline

In [27]:
remaining_missing = pd.DataFrame({
    "Training Missing": X_train.isnull().sum(),
    "Test Missing": X_test.isnull().sum()
})

remaining_missing = remaining_missing[
    (remaining_missing["Training Missing"] > 0)
    | (remaining_missing["Test Missing"] > 0)
]

remaining_missing

,Training Missing,Test Missing
children,4,0
country,415,73


## 8. Class-Imbalance Decision

The target variable is moderately imbalanced but not extremely skewed.

No oversampling, undersampling, or SMOTE transformation is applied at this stage.

Synthetic resampling before the chronological split would create leakage, while applying it without first establishing a baseline would make it difficult to determine whether it is actually beneficial.

During model development, performance will therefore be evaluated using multiple metrics including precision, recall, F1-score, ROC-AUC, and the confusion matrix. Class-weight options may also be evaluated for suitable algorithms using training data only.

#### Training target distribution

In [28]:
training_target_summary = pd.DataFrame({
    "Count": y_train.value_counts().sort_index(),
    "Percentage": (
        y_train
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
})

training_target_summary.index = [
    "Not Cancelled (0)",
    "Cancelled (1)"
]

training_target_summary

,Count,Percentage
Not Cancelled (0),58738,61.57
Cancelled (1),36663,38.43


## 9. Define Numerical and Categorical Features

Predictor variables are classified according to their current representation.

Categorical variables will be one-hot encoded.

Numerical variables will be imputed where necessary and scaled using `RobustScaler`, which uses the median and interquartile range and is less influenced by extreme values than standard mean/standard-deviation scaling.

The `children` variable receives a separate most-frequent imputation treatment because it is a discrete count with only a very small number of missing values.

#### Identify predictor groups

In [29]:
categorical_columns = (
    X_train
    .select_dtypes(include=["object"])
    .columns
    .tolist()
)

numeric_columns = (
    X_train
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

numeric_columns_without_children = [
    column
    for column in numeric_columns
    if column != "children"
]

print(
    "Categorical features:",
    len(categorical_columns)
)

print(
    "Numerical features:",
    len(numeric_columns)
)

print("\nCategorical columns:")
print(categorical_columns)

print("\nNumerical columns:")
print(numeric_columns)

Categorical features: 10
Numerical features: 22

Categorical columns:
['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'customer_type', 'booking_month']

Numerical columns:
['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_nights', 'total_guests', 'has_children', 'has_agent', 'has_company', 'is_zero_guest', 'is_zero_night']


## 10. Build the Preprocessing Pipeline

Three preprocessing branches are used:

1. General numerical features:
   - median imputation as a safety measure;
   - robust scaling.

2. `children`:
   - most-frequent imputation;
   - robust scaling.

3. Categorical features:
   - missing values represented as `Unknown`;
   - one-hot encoding;
   - previously unseen test categories ignored safely.

The preprocessing transformations will be fitted only on training data.

#### Numerical pipeline

In [30]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            RobustScaler()
        )
    ]
)

#### Children pipeline

In [31]:
children_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "scaler",
            RobustScaler()
        )
    ]
)

#### Categorical pipeline

In [32]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

#### Combine preprocessing branches

In [33]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_columns_without_children
        ),

        (
            "children",
            children_pipeline,
            ["children"]
        ),

        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )
    ]
)

preprocessor

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', RobustScaler())]),
                                 ['lead_time', 'arrival_date_year',
                                  'arrival_date_week_number',
                                  'arrival_date_day_of_month',
                                  'stays_in_weekend_nights',
                                  'stays_in_week_nights', 'adults', 'babies',
                                  'is_repeated_guest', 'previous_cancellations',
                                  'previous_bookings...
                                                 ('scaler', RobustScaler())]),
                                 ['children']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Unknown',
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['hotel', 'arrival_date_month', 'meal',
                                  'country', 'market_segment',
                                  'distribution_channel', 'reserved_room_type',
                                  'deposit_type', 'customer_type',
                                  'booking_month'])])

## 11. Verify the Preprocessing Pipeline

The preprocessing pipeline is temporarily fitted on the complete chronological training set to confirm that it works correctly.

This transformed output is used only for verification in this notebook.

During model development, preprocessing will be placed inside each machine-learning pipeline so that cross-validation fits the preprocessing steps separately within each training fold. This prevents validation-fold information from influencing imputation, scaling, or categorical encoding.

#### Fit on training data and transform both sets

In [34]:
X_train_processed = (
    preprocessor
    .fit_transform(X_train)
)

X_test_processed = (
    preprocessor
    .transform(X_test)
)

print(
    "Processed training shape:",
    X_train_processed.shape
)

print(
    "Processed test shape:",
    X_test_processed.shape
)

Processed training shape: (95401, 251)
Processed test shape: (23989, 251)


#### Get transformed feature names

In [35]:
processed_feature_names = (
    preprocessor
    .get_feature_names_out()
)

print(
    "Number of processed features:",
    len(processed_feature_names)
)

print("\nFirst 30 processed features:")

for feature in processed_feature_names[:30]:
    print(feature)

Number of processed features: 251

First 30 processed features:
numeric__lead_time
numeric__arrival_date_year
numeric__arrival_date_week_number
numeric__arrival_date_day_of_month
numeric__stays_in_weekend_nights
numeric__stays_in_week_nights
numeric__adults
numeric__babies
numeric__is_repeated_guest
numeric__previous_cancellations
numeric__previous_bookings_not_canceled
numeric__adr
numeric__required_car_parking_spaces
numeric__total_of_special_requests
numeric__total_nights
numeric__total_guests
numeric__has_children
numeric__has_agent
numeric__has_company
numeric__is_zero_guest
numeric__is_zero_night
children__children
categorical__hotel_City Hotel
categorical__hotel_Resort Hotel
categorical__arrival_date_month_April
categorical__arrival_date_month_August
categorical__arrival_date_month_December
categorical__arrival_date_month_February
categorical__arrival_date_month_January
categorical__arrival_date_month_July


#### Check for non-finite values

In [36]:
from scipy import sparse


def count_non_finite(matrix):

    if sparse.issparse(matrix):
        return int(
            (~np.isfinite(matrix.data)).sum()
        )

    return int(
        (~np.isfinite(matrix)).sum()
    )


print(
    "Non-finite values in processed training data:",
    count_non_finite(X_train_processed)
)

print(
    "Non-finite values in processed test data:",
    count_non_finite(X_test_processed)
)

Non-finite values in processed training data: 0
Non-finite values in processed test data: 0


#### Verify row counts survived transformation

In [37]:
assert (
    X_train_processed.shape[0]
    == len(y_train)
)

assert (
    X_test_processed.shape[0]
    == len(y_test)
)

print(
    "Processed predictors and targets "
    "have matching row counts."
)

Processed predictors and targets have matching row counts.


## 12. Save Reproducible Intermediate Splits

The engineered but unencoded training and test datasets are saved under `data/processed/`.

These files are reproducible outputs rather than raw source data. They allow later model-development notebooks to reuse the same chronological split.

The final model notebook will still construct preprocessing inside each machine-learning pipeline to ensure leakage-safe cross-validation.

### Save engineered splits

In [38]:
os.makedirs(
    "data/processed",
    exist_ok=True
)

X_train.to_csv(
    "data/processed/X_train_engineered.csv",
    index=False
)

X_test.to_csv(
    "data/processed/X_test_engineered.csv",
    index=False
)

y_train.to_frame().to_csv(
    "data/processed/y_train.csv",
    index=False
)

y_test.to_frame().to_csv(
    "data/processed/y_test.csv",
    index=False
)

print(
    "Engineered train/test splits "
    "saved successfully."
)

Engineered train/test splits saved successfully.


## 13. Preprocessing Summary

The preprocessing stage produced the following decisions:

- A reconstructed booking date was created from arrival date and lead time.
- The data was divided chronologically into an approximately 80% training set and 20% final test set.
- Ambiguous cleaning decisions were investigated using training data only.
- Exact duplicate records were retained because no unique reservation ID proves that they are erroneous.
- Zero-guest, zero-night, large-group, negative-ADR, and very-high-ADR observations were retained because there was insufficient evidence to classify them as incorrect.
- Missing `country` values are represented as `Unknown`.
- Missing `children` values are imputed using the most frequent training value.
- Raw `agent` and `company` identifiers were replaced by presence indicators.
- Outcome-related and timing-sensitive variables were removed to reduce data leakage.
- Additional booking-level features were engineered.
- Numerical variables are robustly scaled.
- Categorical variables are one-hot encoded with protection against unseen categories.
- No class-resampling technique was applied before establishing baseline model performance.
- The final chronological test set was not used to learn preprocessing parameters.
- During model development, the preprocessor will be embedded inside model pipelines to preserve leakage-safe cross-validation.

### Final integrity checks

In [39]:
print("Raw dataset shape:")
print(df.shape)

print("\nChronological split:")
print("Training:", len(X_train))
print("Test:", len(X_test))

print("\nFinal predictor count before encoding:")
print(X_train.shape[1])

print("\nFinal predictor count after encoding:")
print(X_train_processed.shape[1])

print("\nTarget:")
print(y_train.name)

print("\nLeakage columns present:")
print([
    col
    for col in [
        "reservation_status",
        "reservation_status_date",
        "assigned_room_type",
        "booking_changes",
        "days_in_waiting_list"
    ]
    if col in X_train.columns
])

Raw dataset shape:
(119390, 32)

Chronological split:
Training: 95401
Test: 23989

Final predictor count before encoding:
32

Final predictor count after encoding:
251

Target:
is_canceled

Leakage columns present:
[]


### Final Data-Integrity Check

The original raw dataframe remains unchanged.

All modelling decisions were performed on separate training and test copies. The final test set remained chronologically later than the training data and was not used to fit imputation, scaling, or encoding parameters.

Notebook 02 therefore produces a leakage-aware and reproducible dataset preparation workflow suitable for model development in Notebook 03.